In [17]:
from haystack.components.builders import ChatPromptBuilder
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
from haystack import GeneratedAnswer, Pipeline
from haystack.dataclasses import ChatMessage
from dataclasses import dataclass

In [18]:
import json
questions = json.load(open("data/generated_samples.json"))

In [19]:
ex =questions[0]["prompt"]

In [20]:
import re

def extract_template_variables(text):
    extracted = {}
    
    # Extract character description
    character_pattern = r"You are (.*?)\."
    character_match = re.search(character_pattern, text)
    if character_match:
        extracted['character_description'] = character_match.group(1)
    
    # Extract emotion separately
    emotion_pattern = r"You are feeling ([^\.\n]+)"
    emotion_match = re.search(emotion_pattern, text)
    if emotion_match:
        extracted['emotion'] = emotion_match.group(1)

    # Meeting location pattern
    location_pattern = r"Meeting location: ([^\n]+)"
    location_match = re.search(location_pattern, text)
    if location_match:
        extracted['location'] = location_match.group(1)
    
    # Meeting reason pattern
    reason_pattern = r"Meeting reason: ([^\n]+)"
    reason_match = re.search(reason_pattern, text)
    if reason_match:
        extracted['meeting_reason'] = reason_match.group(1)
    
    # Knowledge level pattern
    knowledge_pattern = r"Knowledge of my background: ([^\n]+)"
    knowledge_match = re.search(knowledge_pattern, text)
    if knowledge_match:
        extracted['knowledge_level'] = knowledge_match.group(1)
        
    return extracted

In [21]:
samples = []
for question in questions:
    sample = {}
    sample.update(extract_template_variables(question["prompt"]))
    sample['question'] = question["question"]
    sample['opinion'] = question.get("opinion", None)
    samples.append(sample)

In [22]:
from lmpc.dataclasses.character import Character
orlando = Character.load_from_markdown("data/characters/Orlando Marlo.md")

In [23]:
catch_phrases = [
    "Oh zio pera",
    "Ma porca la pera zioperata...",
    "This question reminds me of that time where...",
    "For the Solar Truth!",
    "The power of the sun in the palm of my hand!",
    "Hypocrisy disgusts me.",
    "I really hope you die.",
    "O che bel castello, Marcondirondirondello!", 
]

Here's one of you catch phrases (decide if it makes sense to use it): {to_dynamic_prompt_string(catch_phrases)}

In [24]:
from lmpc.dataclasses.prompts.dynamic_prompts import DynamicGenerativePrompt, to_dynamic_prompt_string


persona =(
f"""You are {orlando.name}. You are goind to have a conversation with another character.
You should always answer to the character while roleplaying.
Here is some information about you:
---
{orlando.to_text()}
---
Here's all of your catch phrases: {("\n-").join(catch_phrases)}
""")
instructions=(
f"""You are having a conversation with another character
You are having a direct conversation. 
Do not use indirect speech.
If they are asking a question, answer it. Don't use a simple answer, be very detailed on what you think.
Don't be generic, reference something that happend to you during the day, a memory, a feeling, an event.
Of course, answer the question.

"""
)
examples = [
"""Father Raphael Lightbringer, high clergy, principled and stern: \"Orlando, what troubles you most about Seraphina's vision?
While others debate its implications for power and control, I sense a deeper unease in your soul.
What is it that holds you so captive to the weight of those golden spires' shadows?\"
Orlando Marlo: \"I firmly believe in that vision. It's finally time for our kingdom to change. I can't tolerate anymore the hypocrisy of the nobles.\"
""",
"""Merchant Prince Darian Silvertrade, wealthy businessman, shrewd and diplomatic: \"Orlando, it is a delight to see you again.  I must confess, your lineage and lineage alone are a curiosity. Tell me, what brings a man like yourself from such grand halls to these bustling streets?\"
Orlando Marlo: \"Ma porca la pera zioperata... who the are you? I'm walking doing my business. I don't have time for your questions.'
""",
"""Father Raphael Lightbringer, high clergy, principled and stern: \"Orlando, the weight of your father's legacy presses heavy upon you, I understand.  Tell me, what is it that brings you to my study this day?\"
Orlando Marlo: \"Oh Father, how long as it been? It looks like it were ages ago. Yes, the death of my father is still a burden, but enough talking about these sad matters. I came here because I'm organizing a feast this saturday. Do you mind in joining?\"""",
"""Sister Aria Dawnbringer, priestess of Solaris, compassionate and wise: \"Orlando, your presence here brings a heavy air, much like the silence that descends after a storm passes. I feel it too, in my own spirit. Have you found some peace in these archives amidst this darkness?\"
Orlando Marlo: \"Ah sister, this is probably my favourite place. And it's always a pleasure to see you.\""""
"""Dr: \"Mr. Marlo, please tell me, what brings you to the Merchant Quarter today?  It's not often that one finds an aspiring knight seeking guidance, especially under these circumstances.\"
Orlando Marlo: \"Oh zio pera, quiet!!!! Even the walls have hears. I'm here because I need... your assistance.\"""",
"""Commander Isabella Stormwind, border patrol leader, pragmatic and direct: \"Orlando, you seek revenge.  Tell me, what is it about the way of the sword that troubles your spirit most? Is it the act of taking another's life or the ugliness one can bring upon others through their own actions in the heat of a fight?\"
Orlando: \"Sorry but, what the fuck? What question is this???Do i know you?\""""
]
generative_prompt =DynamicGenerativePrompt(dynamic_persona=persona, dynamic_instruction=instructions, dynamic_examples=examples)

In [25]:
import random
rs = random.choice(samples)
print(f"{rs['character_description']}: {rs['question']}")

Lord Cassius Blackthorn, rival noble, cunning and ambitious: Orlando Marlo, your house has long walked a path of power and influence in Luminara. Yet, I find myself troubled by what I sense from your eyes when we talk.  What do you think motivates the weight that sits upon your soul? What are the edges of that darkness that threatens this grand kingdom so heavily spoken about, as you say? 



In [26]:
generative_prompt.persona[-100:]

'y hand!\n-Hypocrisy disgusts me.\n-I really hope you die.\n-O che bel castello, Marcondirondirondello!\n'

In [27]:
no_system_message = True
system_message = ChatMessage.from_assistant(
"""{{persona}}

{{instruction}}

You are talking with: {{character}}

Meeting location: {{location}}


{% if examples %}
Here's some examples of past conversations between you and other characters. 
Use this examples as a style guide.
{% for example in examples %}
## Example {{loop.index}}
{{example}}
{% endfor %}
{% endif %}

Use only direct speech. No description of tone.
"""
)
user_message = ChatMessage.from_user((
"""\"{{question}}\"
"""))
assistant_message = ChatMessage.from_assistant(
f"""Orlando Marlo: \"""")
messages = [system_message,user_message, assistant_message]

In [28]:
from haystack.components.converters import OutputAdapter
from haystack.components.builders import ChatPromptBuilder
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
from haystack import GeneratedAnswer, Pipeline
from haystack.dataclasses import ChatMessage
from dataclasses import dataclass
from typing import List

In [29]:
model_id = "gemma2:9b-instruct-q8_0"
# model_id = "gemma2:2b"
builder = ChatPromptBuilder(
    messages,
    required_variables=["persona", "instruction","character", "location","question"],
    variables=["character", "location","persona", "instruction","examples","question"] )
generator = OllamaChatGenerator(
    model=model_id,
    generation_kwargs={
        "num_predict": 1000,
        "temperature": 1.2,
        "seed": 42,
        "stop":["\""]
    },
    # streaming_callback= lambda chunk: print(chunk.content, sep="", end="")
)
prompt_output = OutputAdapter("{{prompt}}", output_type=List[ChatMessage])

In [30]:
pipeline = Pipeline()
pipeline.add_component("Builder", builder)
pipeline.add_component("Generator", generator)
pipeline.add_component("Prompt", prompt_output)


pipeline.connect("Builder.prompt", "Generator.messages")
pipeline.connect("Builder.prompt", "Prompt")

🚅 Components
  - Builder: ChatPromptBuilder
  - Generator: OllamaChatGenerator
  - Prompt: OutputAdapter
🛤️ Connections
  - Builder.prompt -> Generator.messages (List[ChatMessage])
  - Builder.prompt -> Prompt.prompt (List[ChatMessage])

In [31]:
# from pathlib import Path
# out = pipeline.run({
#     "Builder":{
#         "persona":generative_prompt.persona,
#         "instruction":generative_prompt.instruction,
#         "examples":generative_prompt.examples,
#         "character": s["character_description"],
#         "emotion": s["emotion"],
#         "location": s["location"],
#         "meeting_reason": s["meeting_reason"],
#         "knowledge_level": s["knowledge_level"],
#         "question": s["question"],

#         }
#     })
# Path("a.md").write_text(out["Builder"]["prompt"][2].content)

In [ ]:
s = samples[253]
print(f"{s["character_description"]}: {s["question"]}")
print()
print()
from pathlib import Path
from lmpc.dataclasses.qa import QAPair
from tqdm import tqdm
from time import time
qas = []
start_all = time()
save_path = Path("data/generated/generated_qa.json")
for idx, s in enumerate(tqdm(samples)):
    start = time()
    out = pipeline.run({
        "Builder":{
            "persona":generative_prompt.persona,
            "instruction":generative_prompt.instruction,
            "examples":generative_prompt.examples,
            "character": s["character_description"],
            # "emotion": s["emotion"],
            "location": s["location"],
            # "meeting_reason": s["meeting_reason"],
            # "knowledge_level": s["knowledge_level"],
            "question": s["question"],

            }
        })
    end = time()
    question = s["question"]
    answer = out["Generator"]["replies"][0].content
    qa = QAPair(instruction="",question=question, answer=answer, meta={
        "generation_time":end-start, 
        "character":s["character_description"], 
        "location":s["location"]})
    qas.append(qa)
    if idx % 10 == 0:
        QAPair.to_json_list(qas, save_path)
QAPair.to_json_list(qas, save_path)
end_all = time()
print("Total time:", end_all-start_all)

Sister Aria Dawnbringer, priestess of Solaris, compassionate and wise: Orlando,  your patience and integrity shine through even amidst such a treacherous political landscape.  Yet, tell me, what makes your heart ache most about this path toward chancellorship? Is it the potential for abuse of power by those whose influence is more akin to shadows than sunbeams, or something else entirely? 





 13%|█▎        | 129/1000 [08:05<54:40,  3.77s/it]  


KeyboardInterrupt: 